In [1]:
import torch
import pandas as pd
import fitz
from tqdm.auto import tqdm
from spacy.lang.en import English 


In [2]:
pdf_path = "D:\\Projects\\Local_Mini_RAG\\data\\human-nutrition-text.pdf"

def text_formatter(text: str) -> str:
    cleaned_text = text.replace("\n", " ").strip()
    return cleaned_text

def open_and_read_pdf(pdf_path: str) -> list[dict]:
    doc = fitz.open(pdf_path)
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc)):  # iterate the document pages
        text = page.get_text()  # get plain text encoded as UTF-8
        text = text_formatter(text)
        pages_and_texts.append({"page_number": page_number - 41,  # adjust page numbers since our PDF starts on page 42
                                "page_char_count": len(text),
                                "page_word_count": len(text.split(" ")),
                                "page_sentence_count_raw": len(text.split(". ")),
                                "page_token_count": len(text) / 4,  # 1 token = 4 chars
                                "text": text})
    return pages_and_texts

pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[200]


0it [00:00, ?it/s]

{'page_number': 159,
 'page_char_count': 1762,
 'page_word_count': 303,
 'page_sentence_count_raw': 15,
 'page_token_count': 440.5,
 'text': 'thermoregulation is to balance heat gain with heat loss and body  water plays an important role in accomplishing this. Human life  is supported within a narrow range of temperature, with the  temperature set point of the body being 98.6°F (37°C). Too low or  too high of a temperature causes enzymes to stop functioning and  metabolism is halted. At 82.4°F (28°C) muscle failure occurs and  hypothermia sets in. At the opposite extreme of 111.2°F (44°C) the  central nervous system fails and death results. Water is good at  storing heat, an attribute referred to as heat capacity and thus helps  maintain the temperature set point of the body despite changes in  the surrounding environment.  There are several mechanisms in place that move body water  from place to place as a method to distribute heat in the body and  equalize body temperature (Figure 3.

In [40]:
nlp = English()

nlp.add_pipe("sentencizer")

doc = nlp("This is a sentence. This is another sentence. And this is yet another sentence.")

assert len(list(doc.sents)) == 3

list(doc.sents)

[This is a sentence.,
 This is another sentence.,
 And this is yet another sentence.]

In [41]:
for item in tqdm(pages_and_texts):
    # split the text into sentences using spacy
    item["sentences"] = list(nlp(item["text"]).sents)
    # convert the sentences to strings
    item["sentences"] = [str(sentence) for sentence in item["sentences"]]
    # count the number of sentences in the page
    item["page_sentence_count_spacy"] = len(item["sentences"])

  0%|          | 0/1208 [00:00<?, ?it/s]

In [42]:
import random
random.sample(pages_and_texts, 1)

[{'page_number': 969,
  'page_char_count': 1329,
  'page_word_count': 211,
  'page_sentence_count_raw': 13,
  'page_token_count': 332.25,
  'text': 'progresses, the amount of red blood cells will increase to catch up  with the total blood volume.  Vitamin D and Calcium  Vitamin D regulates the calcium and phosphorus absorption and  metabolism and plays a key role in maintaining optimal bone health.  There is also growing evidence that vitamin D is important for  other aspect of athletic performance such as injury prevention,  rehabilitation, and muscle metabolism. Individuals who primarily  practice indoors are at a larger risk for a vitamin D deficiency and  should ensure they are consuming foods high in vitamin D to  maintain sufficient vitamin D status.9  Calcium is especially important for the growth, maintenance, and  repair of bone tissue. Low calcium intake occurs in athletes with  RED-S, menstrual dysfunction, and those who avoid dairy products.  A diet inadequate in calcium in

In [43]:
df = pd.DataFrame(pages_and_texts)

In [44]:
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,198.30,9.97,287.00,10.32
std,348.86,560.38,95.76,6.19,140.10,6.30
min,-41.00,0.00,1.00,1.00,0.00,0.00
25%,260.75,762.00,134.00,4.00,190.50,5.00
50%,562.50,1231.50,214.50,10.00,307.88,10.00
75%,864.25,1603.50,271.00,14.00,400.88,15.00
max,1166.00,2308.00,429.00,32.00,577.00,28.00


In [45]:
num_sentence_chunk_size = 10

def split_list(input_list: list[str],
                slice_size: int= num_sentence_chunk_size) -> list[list[str]]:
    
    return[input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]

test_list = list(range(25))
split_list(test_list)

[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
 [10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
 [20, 21, 22, 23, 24]]

In [46]:
for item in tqdm(pages_and_texts):
    # split the sentences into chunks of num_sentence_chunk_size
    item["sentence_chunks"] = split_list(input_list=item["sentences"], slice_size=num_sentence_chunk_size)
    item["sentence_chunk_count"] = len(item["sentence_chunks"])


  0%|          | 0/1208 [00:00<?, ?it/s]

In [47]:
random.sample(pages_and_texts, 1)

[{'page_number': 311,
  'page_char_count': 787,
  'page_word_count': 145,
  'page_sentence_count_raw': 5,
  'page_token_count': 196.75,
  'text': 'Image by  Openstax  Biology / CC  BY 4.0  Interestingly, some naturally occurring trans fats do not pose the  same health risks as their artificially engineered counterparts.  These trans fats are found in ruminant animals such as cows, sheep,  and goats, resulting in trans fatty acids being present in our meat,  milk, and other dairy product supply. Reports from the US  Department of Agriculture (USDA) indicate that these trans fats  comprise 15 to 20 percent of the total trans-fat intake in our diet.  While we know that trans fats are not exactly harmless, it seems  that any negative effect naturally occurring trans fats have are  counteracted by the presence of other fatty acid molecules in these  animal products, which work to promote human health.  How Lipids Work  |  311',
  'sentences': ['Image by  Openstax  Biology / CC  BY 4.0  Inte

In [48]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)


,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy,sentence_chunk_count
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,198.30,9.97,287.00,10.32,1.53
std,348.86,560.38,95.76,6.19,140.10,6.30,0.64
min,-41.00,0.00,1.00,1.00,0.00,0.00,0.00
25%,260.75,762.00,134.00,4.00,190.50,5.00,1.00
50%,562.50,1231.50,214.50,10.00,307.88,10.00,1.00
75%,864.25,1603.50,271.00,14.00,400.88,15.00,2.00
max,1166.00,2308.00,429.00,32.00,577.00,28.00,3.00


In [49]:
import re

pages_and_chunks = []

for item in tqdm(pages_and_texts):
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}
        chunk_dict["page_number"] = item["page_number"]

        # join the sentences in the chunk back into a single string
        joined_sentence_chunk = " ".join(sentence_chunk).replace("  ", " ").strip()
        # add space after period if there isn't one already
        joined_sentence_chunk = re.sub(r"\.([A-Z])", r". \1", joined_sentence_chunk)
        # add the joined sentence chunk to the chunk dict
        chunk_dict["sentence_chunk"] = joined_sentence_chunk

        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len(joined_sentence_chunk.split(" "))
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4
        pages_and_chunks.append(chunk_dict)

len(pages_and_chunks)

  0%|          | 0/1208 [00:00<?, ?it/s]

1843

In [50]:
random.sample(pages_and_chunks, 1)

[{'page_number': 926,
  'sentence_chunk': 'Vision Problems Many older people suffer from vision problems and a loss of vision. Age-related macular degeneration is the leading cause of blindness in Americans over age sixty.6 This disorder can make food planning and preparation extremely difficult and people who suffer from it often must depend on caregivers for their meals. Self-feeding also may be difficult if an elderly person cannot see his or her food clearly. Friends and family members can help older adults with shopping and cooking. Food-assistance programs for older adults (such as Meals on Wheels) can also be helpful. Diet may help to prevent macular degeneration. Consuming colorful fruits and vegetables increases the intake of lutein and zeaxanthin. Several studies have shown that these antioxidants provide protection for the eyes. Lutein and zeaxanthin are found in green, leafy vegetables such as spinach, kale, and collard greens, and also corn, peaches, squash, broccoli, Brus

In [51]:
df = pd.DataFrame(pages_and_chunks)
df.describe().round(2)

,page_number,chunk_char_count,chunk_word_count,chunk_token_count
count,1843.00,1843.00,1843.00,1843.00
mean,583.38,735.14,113.03,183.78
std,347.79,447.64,71.27,111.91
min,-41.00,12.00,3.00,3.00
25%,280.50,315.00,45.00,78.75
50%,586.00,747.00,114.00,186.75
75%,890.00,1119.00,174.00,279.75
max,1166.00,1832.00,298.00,458.00


In [52]:
min_token_length = 30

pages_and_chunks_over_min_token_length = df[df["chunk_token_count"]> min_token_length].to_dict(orient="records")
random.sample(pages_and_chunks_over_min_token_length,k=1)

[{'page_number': 793,
  'sentence_chunk': 'Fats should make up 25 to 35 percent of daily calories, and those calories should come from healthy fats, such as avocados and salmon. It is not recommended for pregnant women to be on a very low-fat diet, since it would be hard to meet the needs of essential fatty acids and fat-soluble vitamins. Fatty acids are important during pregnancy because they support the baby’s brain and eye development. Fluids Fluid intake must also be monitored. According to the IOM, pregnant women should drink 2.3 liters (about 10 cups) of liquids per day to provide enough fluid for blood production. It is also important to drink liquids during and after physical activity or when it is hot and humid outside, to replace fluids lost to perspiration. Pregnancy | 793',
  'chunk_char_count': 751,
  'chunk_word_count': 129,
  'chunk_token_count': 187.75}]

In [54]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2", device="cuda" if torch.cuda.is_available() else "cpu")



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [55]:
%%time

for item in tqdm(pages_and_chunks_over_min_token_length):
    item["embedding"] = embedding_model.encode(item["sentence_chunk"])

  0%|          | 0/1680 [00:00<?, ?it/s]

CPU times: total: 7min 10s
Wall time: 56.2 s


In [56]:
%%time

text_chunk = [item['sentence_chunk'] for item in pages_and_chunks_over_min_token_length]
text_chunk[419]

CPU times: total: 0 ns
Wall time: 587 μs


'often. • Calm your “sweet tooth” by eating fruits, such as berries or an apple. • Replace sugary soft drinks with seltzer water, tea, or a small amount of 100 percent fruit juice added to water or soda water. The Food Industry: Functional Attributes of Carbohydrates and the Use of Sugar Substitutes In the food industry, both fast-releasing and slow-releasing carbohydrates are utilized to give foods a wide spectrum of functional attributes, including increased sweetness, viscosity, bulk, coating ability, solubility, consistency, texture, body, and browning capacity. The differences in chemical structure between the different carbohydrates confer their varied functional uses in foods. Starches, gums, and pectins are used as thickening agents in making jam, cakes, cookies, noodles, canned products, imitation cheeses, and a variety of other foods. Molecular gastronomists use slow- releasing carbohydrates, such as alginate, to give shape and texture to their fascinating food creations. Add

In [57]:
%%time
text_chunk_embedding = embedding_model.encode(text_chunk, batch_size=16, convert_to_tensor=True)

text_chunk_embedding

CPU times: total: 2min 37s
Wall time: 32.3 s


tensor([[ 0.0674,  0.0902, -0.0051,  ..., -0.0221, -0.0232,  0.0126],
        [ 0.0552,  0.0592, -0.0166,  ..., -0.0120, -0.0103,  0.0227],
        [ 0.0280,  0.0340, -0.0206,  ..., -0.0054,  0.0213,  0.0313],
        ...,
        [ 0.0771,  0.0098, -0.0122,  ..., -0.0409, -0.0752, -0.0241],
        [ 0.1030, -0.0165,  0.0083,  ..., -0.0574, -0.0283, -0.0295],
        [ 0.0864, -0.0125, -0.0113,  ..., -0.0522, -0.0337, -0.0299]],
       device='cuda:0')

In [58]:
pages_and_chunks_over_min_token_length[419]

{'page_number': 277,
 'sentence_chunk': 'often. • Calm your “sweet tooth” by eating fruits, such as berries or an apple. • Replace sugary soft drinks with seltzer water, tea, or a small amount of 100 percent fruit juice added to water or soda water. The Food Industry: Functional Attributes of Carbohydrates and the Use of Sugar Substitutes In the food industry, both fast-releasing and slow-releasing carbohydrates are utilized to give foods a wide spectrum of functional attributes, including increased sweetness, viscosity, bulk, coating ability, solubility, consistency, texture, body, and browning capacity. The differences in chemical structure between the different carbohydrates confer their varied functional uses in foods. Starches, gums, and pectins are used as thickening agents in making jam, cakes, cookies, noodles, canned products, imitation cheeses, and a variety of other foods. Molecular gastronomists use slow- releasing carbohydrates, such as alginate, to give shape and texture 

In [59]:
text_chunk_embedding_df = pd.DataFrame(pages_and_chunks_over_min_token_length)

embedding_df_save_path = "D:\\Projects\\Local_Mini_RAG\\text_chunk_embeddings.csv"

text_chunk_embedding_df.to_csv(embedding_df_save_path, index=False)

In [60]:
text_chunk_embedding_df_load = pd.read_csv(embedding_df_save_path)

text_chunk_embedding_df_load.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding
0,-39,Human Nutrition: 2020 Edition UNIVERSITY OF HA...,308,42,77.0,[ 6.74242526e-02 9.02281031e-02 -5.09549724e-...
1,-38,Human Nutrition: 2020 Edition by University of...,210,30,52.5,[ 5.52156046e-02 5.92139363e-02 -1.66167300e-...
2,-37,Contents Preface University of Hawai‘i at Māno...,766,114,191.5,[ 2.79801898e-02 3.39813866e-02 -2.06426680e-...
3,-36,Lifestyles and Nutrition University of Hawai‘i...,942,143,235.5,[ 6.82566911e-02 3.81274968e-02 -8.46854970e-...
4,-35,The Cardiovascular System University of Hawai‘...,998,152,249.5,[ 3.30264494e-02 -8.49766005e-03 9.57159698e-...
